# Part 2: Add structured data from Fabric IQ

Part 2 at Caldova begins when a procurement analyst finds Meridian API Works invoice and quality evidence for lot `AUR-API-7`. The documents identify the lot and its active substance, but you need Fabric IQ to trace how the manufacturer connects to that substance and which Caldova medicinal product uses it.

**🎯 Mission**
- Preserve the purchasing policy file and both **Indexed Knowledge Sources** from Part 1
- Add a **Fabric IQ Ontology Knowledge Source** for manufacturer, active-substance, and medicinal-product relationships
- Build a knowledge base that distinguishes indexed lot evidence from structured Fabric relationships

## The building blocks

In Part 1, your knowledge base queried policy and indexed procurement documents. Now you'll add a **Fabric Ontology Knowledge Source** that connects to structured operational data in Microsoft Fabric.

- **Fabric Ontology Knowledge Source**: Connects to a Fabric workspace ontology. The knowledge base issues structured queries against entities and relationships at runtime.
- **Delegated token**: This notebook uses `InteractiveBrowserCredential` to sign in the current user and obtain a delegated token for Foundry IQ (Azure AI Search). Search uses that delegated user identity when querying protected sources such as Fabric Ontology.

## Step 1: Set up environment variables

Load the configuration for your Foundry IQ (Azure AI Search), Azure OpenAI, and Fabric resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

SUPPLIER_EVIDENCE_INDEX = "supplier-evidence"
SOURCING_DOCUMENTS_INDEX = "sourcing-documents"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## Step 2: Login and acquire delegated token

In order for the knowledge base retrieval call to use your identity for query source access, it needs an OAuth token for a signed-in user who has access to the Fabric workspace.

Run the cell below to sign in using the credentials in the instructions sidebar. If `FABRIC_TENANT_ID` is configured, authentication uses that tenant. Otherwise, it uses `AZURE_TENANT_ID`. A successful sign-in acquires a delegated token for the Foundry IQ (Azure AI Search) scope and saves it for use as `query_source_authorization` during retrieval.

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(
    tenant_id=os.getenv("FABRIC_TENANT_ID", AZURE_TENANT_ID)
)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## Step 3: Create the indexed and Fabric knowledge sources

Reuse the direct policy file source from Part 1, recreate the two indexed source definitions, and add Fabric IQ:

* `file-docs-knowledge-source`: Caldova's uploaded purchasing policy
* `supplier-evidence-knowledge-source`: Supplier invoices, shipment evidence, production yield records, and quality records
* `sourcing-documents-knowledge-source`: RFP responses, agreements, amendments, and purchase orders
* `fabric-ontology-knowledge-source`: Structured manufacturer, active-substance, and medicinal-product relationships in Fabric IQ

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"
SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME = "supplier-evidence-knowledge-source"
SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "sourcing-documents-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    FILE_KNOWLEDGE_SOURCE_NAME,
    SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (
        SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
        SUPPLIER_EVIDENCE_INDEX,
        "Caldova supplier invoices, shipment evidence, production yield records, and quality records",
    ),
    (
        SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        SOURCING_DOCUMENTS_INDEX,
        "Caldova RFP responses, agreements, amendments, and purchase orders",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Caldova Fabric ontology for manufacturer, active-substance, and medicinal-product relationships",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")


## Step 4: Create the multi-source Fabric knowledge base

This knowledge base preserves the purchasing policy file and both indexed evidence sources, then adds Fabric IQ for structured relationships.

The knowledge base uses `outputMode=answerSynthesis` and `retrievalReasoningEffort=low`. Its retrieval instructions establish an important boundary: indexed documents provide the lot evidence, while Fabric IQ provides manufacturer-to-substance and substance-to-product relationships.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "caldova-fabric-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova knowledge base combining purchasing policy, indexed procurement evidence, and Fabric relationships",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use indexed supplier evidence for invoice and quality facts. Use Fabric Ontology only for manufacturer-to-substance and substance-to-product relationships. Clearly label which source supports each statement.",
    answer_instructions="Treat fabricRawData as the authoritative data source for manufacturer, active-substance, and medicinal-product relationships. Report what answers you found in fabricRawData.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 5: Query the knowledge base

Ask a question that begins with Meridian API Works invoice and quality evidence for `AUR-API-7`, then follows structured Fabric IQ relationships to a Caldova medicinal product.

The answer should identify which facts came from indexed evidence and which relationships came from Fabric IQ. A relationship that does not appear in the ontology must not be inferred.

⏱️ The retrieval takes about 40 seconds because the request includes Fabric IQ.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = (
    "Which products use Meridian API Works' active substances, and which product is "
    "their invoiced AUR-API-7 lot used in?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        FileKnowledgeSourceParams(
            knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            reranker_threshold=0.0,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

For this activity log, you will see a "fabricOntology" step for the call made to Fabric IQ, along with "searchIndex" steps for each query sent to a search index.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

Indexed references have `type: "searchIndex"` and expose Caldova citation metadata such as `title`, `blob_path`, and page numbers. Fabric IQ references have `type: "fabricOntology"`, with `fabricAnswer` for the synthesized ontology response and `fabricRawData` for structured results.

Use both reference types to verify that invoice and quality claims come from indexed evidence, while manufacturer-to-substance and substance-to-product links come from Fabric IQ.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which indexed documents support the invoiced `AUR-API-7` lot and its quality evidence?
2. Which Fabric IQ results establish the manufacturer-to-substance relationship?
3. Which Fabric IQ results establish the substance-to-product relationship?
4. Does every statement in the answer preserve the boundary between document evidence and structured relationships?

## ✅ Mission complete

**What you built:**

* ✓ **Inherited procurement sources**: Caldova's purchasing policy plus supplier evidence and sourcing documents remain available in the knowledge base.
* ✓ **Fabric IQ Ontology Knowledge Source**: Structured manufacturer, active-substance, and medicinal-product relationships can be queried at runtime.
* ✓ **Evidence-aware retrieval**: Indexed documents support the `AUR-API-7` lot facts, while Fabric IQ supports the relationships used to identify the medicinal product.

➡️ Continue to: [Part 3: Add Work IQ](part3-work-iq-to-kb.ipynb).